In [0]:
from pyspark.sql import Row

# Create initial DataFrame with sample data
data = [Row(id=1, name='Alice', age=30), Row(id=2, name='Bob', age=25), Row(id=3, name='Charlie', age=35)]
df = spark.createDataFrame(data)

In [0]:
df.write.saveAsTable("my_files.my_schema.people_delta_demo")

In [0]:
from pyspark.sql import Row

# Create initial DataFrame with sample data
data = [Row(id=1, name='Alice', age=30), Row(id=2, name='Bob', age=25), Row(id=3, name='Charlie', age=35)]
df = spark.createDataFrame(data)

# Overwrite and create a managed Delta table with the DataFrame in the training.delta_demo schema
df.write.format('delta').mode('overwrite').saveAsTable('my_files.my_schema.people_delta_demo')

# Read the Delta table to verify creation and display its contents
delta_df = spark.read.table('my_files.my_schema.people_delta_demo')
display(delta_df)

In [0]:
#writing the data to a volume
df.write.format('delta').mode('overwrite').save("/Volumes/my_files/my_schema/test_volume/DL_Demo")

In [0]:
#Querying the location
df2=spark.read.load("/Volumes/my_files/my_schema/test_volume/DL_Demo")
display(df2)

In [0]:
%sql
describe history my_files.my_schema.people_delta_demo

--displays the version and history of the data

In [0]:
from pyspark.sql.functions import lit

# Create DataFrame with new rows and add 'country' column
data = [Row(id=7, name='Tom'), Row(id=8, name='Herry'), Row(id=9, name='Jerry')]
df = spark.createDataFrame(data).withColumn("country", lit("USA"))
display(df)

In [0]:
# Append new rows to Delta table, merging schema to add 'country' column if not present
df.write.format('delta').mode('append')\
     .option('mergeSchema', 'true')\
    .saveAsTable('my_files.my_schema.people_delta_demo')

# Read and display updated Delta table to verify schema evolution and new data
result_df = spark.read.table('my_files.my_schema.people_delta_demo')
display(result_df)

In [0]:
%sql
describe history my_files.my_schema.people_delta_demo

--displays the version and history of the data

In [0]:
%sql
SELECT * FROM my_files.my_schema.people_delta_demo VERSION AS OF 2

-- grabs the second version of the data

In [0]:
%sql
SELECT * FROM my_files.my_schema.people_delta_demo TIMESTAMP AS OF '2026-01-17T05:05:13.000+00:00'

-- grabs the second version of the data

In [0]:
%sql
RESTORE my_files.my_schema.people_delta_demo VERSION AS OF 2

-- Restores the table to the second version of the data

In [0]:
%sql
SELECT * FROM my_files.my_schema.people_delta_demo

-- queries the data

In [0]:
%sql
describe history my_files.my_schema.people_delta_demo

--displays the version and history of the data

In [0]:
from pyspark.sql.functions import lit

# Prepare new data for merge
data = [Row(id=1, name='Tom', age=28), Row(id=7, name='Herry', age=27), Row(id=8, name='Jerry', age=37)]
df = spark.createDataFrame(data)
display(df)

In [0]:
# Create a temporary view for the new data
df.createOrReplaceTempView("updates")

# Perform MERGE INTO for insert and update based on id column
merge_sql = """
MERGE INTO my_files.my_schema.people_delta_demo AS target
USING updates AS source
ON target.id = source.id
WHEN MATCHED THEN
  UPDATE SET target.name = source.name, target.age = source.age
WHEN NOT MATCHED THEN
  INSERT (id, name, age) VALUES (source.id, source.name, source.age)
"""

spark.sql(merge_sql)

# Remove country column and keep only id, name, and age
result_df = spark.read.table('my_files.my_schema.people_delta_demo').select("id", "name", "age")
display(result_df)

In [0]:
%sql
describe history my_files.my_schema.people_delta_demo

--displays the version and history of the data